# Predictive and Generative Modeling of Therapeutic Antibody Developability
## Generative Pipeline: Flow Matching for HCDR3 Infilling

**Pipeline:**
1. Encode PROPHET-Ab antibodies with frozen AbLang2 → embeddings
2. Fit ridge regression models predicting HIC & AC-SINS from embeddings
3. Train continuous OT-CFM flow matching model on OAS paired sequences
4. At inference: mask HCDR3, infill via beam-guided Euler ODE (k=5 beams)
5. Evaluate: CamSol / SAP (inline); ΔΔG via IgFold+FoldX (optional)

**Diffusion Baseline (Section 13):** EvoDiff OA-DM-38M

## Section 0 — Install Dependencies

> **Run this cell once, then restart the runtime (Runtime → Restart runtime).**  
> Do NOT install PyTorch — Colab already has a CUDA-compatible build.

**Why the special evodiff install:** evodiff 1.1.2 pins `numpy<2.0` in its metadata,
which would downgrade Colab's numpy 2.x and break pandas/scipy with a binary
incompatibility error (`numpy.dtype size changed`).  
We install evodiff with `--no-deps` to skip that constraint, manually install its
one missing dependency (`sequence-models`), then force numpy back to 2.x.

In [1]:
# ── Step 1: standard packages (no numpy conflicts) ────────────────────────────
# Do NOT reinstall torch — Colab already has the right CUDA version.
!pip install -q ablang2==0.2.1
!pip install -q abnumber==0.4.4
!pip install -q torchdiffeq==0.2.5

# IgFold: needed only for optional 3D ΔΔG evaluation
!pip install -q igfold==0.4.0

# ── Step 2: evodiff — install WITHOUT deps to skip its numpy<2.0 pin ─────────
# evodiff metadata has: Requires-Dist: numpy<2.0,>=1.25
# Installing with --no-deps prevents pip from downgrading numpy.
!pip install -q --no-deps evodiff==1.1.2

# Manually install evodiff's only missing dep (sequence-models has no numpy pin)
!pip install -q sequence-models==1.8.0

# ── Step 3: force numpy back to 2.x (must be last) ───────────────────────────
# This is the critical step: it undoes any numpy downgrade that may have
# slipped in from transitive dependencies, and restores Colab compatibility.
!pip install -q 'numpy>=2.0'

print("\n✓ Done. NOW RESTART THE RUNTIME before running any further cells.")
print("  Runtime → Restart runtime  (do not skip this step)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.3/800.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
evodiff 1.1.2 requires blosum, which is

## Section 1 — Imports & Config

In [3]:
import os, math, random, warnings, pickle
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

import ablang2
from abnumber import Chain as AbnumberChain

try:
    from igfold import IgFoldRunner
    IGFOLD_AVAILABLE = True
except ImportError:
    IGFOLD_AVAILABLE = False
    print("[INFO] igfold not found — 3D ΔΔG will be skipped.")

try:
    from evodiff.pretrained import OA_DM_38M
    from evodiff.conditional_generation import inpaint_simple
    EVODIFF_AVAILABLE = True
except ImportError:
    EVODIFF_AVAILABLE = False
    print("[INFO] evodiff not found — diffusion baseline will be skipped.")

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

AA_VOCAB  = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_VOCAB)}
IDX_TO_AA = {i: aa for aa, i in AA_TO_IDX.items()}
N_AA      = len(AA_VOCAB)   # 20
MAX_CDR3_LEN = 30

[INFO] evodiff not found — diffusion baseline will be skipped.
Device: cuda


/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


## Section 2 — PROPHET-Ab Data Loading

Download supplementary tables from https://doi.org/10.1080/19420862.2025.2593055  
and save as `prophet_ab.csv` with columns:  
`antibody_id`, `vh_sequence`, `vl_sequence`, `hic_rt`, `ac_sins`, `approval`

Until then, a synthetic dataset keeps every downstream cell runnable.

In [2]:
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
drive_file_path = '/content/drive/MyDrive/2026 Spring/BMI 702/project/GDPa1_v1.2_20250814.csv'

df_prophet = pd.read_csv(drive_file_path)
print(f"Loaded  {len(df_prophet)} rows")

print(df_prophet.head())
print("\nColumn dtypes:")
print(df_prophet.dtypes)

Mounted at /content/drive
Loaded  246 rows
  antibody_id antibody_name   Titer  Purity  SEC %Monomer   SMAC    HIC  \
0   GDPa1-001    abagovomab  140.25  98.530        97.010  2.730  2.590   
1   GDPa1-002    abituzumab  193.31  99.825        97.620  2.745  2.545   
2   GDPa1-003   abrezekimab  114.75  98.350        89.055  2.740  2.705   
3   GDPa1-004     abrilumab  327.32  98.575        98.605  2.715  2.565   
4   GDPa1-005    adalimumab  313.39  99.300        96.120  2.705  2.495   

     HAC    PR_CHO    PR_Ova  ...  \
0    NaN  0.337837  0.263108  ...   
1  3.690  0.205246  0.100155  ...   
2    NaN  0.138773  0.101180  ...   
3  1.005  0.000000  0.054971  ...   
4    NaN  0.183387  0.085628  ...   

                                 hc_protein_sequence  \
0  MRAWIFFLLCLAGRALAQVKLQESGAELARPGASVKLSCKASGYTF...   
1  MRAWIFFLLCLAGRALAQVQLQQSGGELAKPGASVKVSCKASGYTF...   
2  MRAWIFFLLCLAGRALAQVTLKESGPVLVKPTETLTLTCTVSGFSL...   
3  MRAWIFFLLCLAGRALAQVQLVQSGAEVKKPGASVKVSCKVSGYTL...   
4  

## Section 3 — Frozen AbLang2 Encoder

Key API facts (verified from ablang2 v0.2.1 source):
- `ablang2.pretrained(model, device=str)` — `device` must be a **string** (`'cuda'` / `'cpu'`)
- Call: `model([(vh, vl), ...], mode='seqcoding')` → numpy array shape `(N, hidden_dim)`
- Call: `model([(vh, vl), ...], mode='rescoding')` → list of numpy arrays, each `(len_vh+len_vl, d)`
- Mask character is `'*'` (tokenizer maps it to mask token id)

In [4]:
print("Loading AbLang2 (downloads ~200 MB on first run)…")
ablang_model = ablang2.pretrained("ablang2-paired", device=str(DEVICE))
ablang_model.freeze()   # sets AbLang.eval()

# Verify mask character from source tokenizer
ABLANG_MASK_CHAR = ablang_model.tokenizer.token_to_aa[
    ablang_model.tokenizer.mask_token
]
assert ABLANG_MASK_CHAR == '*', f"Unexpected mask char: {ABLANG_MASK_CHAR}"
print(f"Mask character confirmed: '{ABLANG_MASK_CHAR}'")

n_params = sum(p.numel() for p in ablang_model.AbLang.parameters())
print(f"AbLang2 parameters: {n_params/1e6:.1f}M  |  all frozen ✓")


def encode_antibody(vh: str, vl: str) -> np.ndarray:
    """
    Encode a VH/VL pair → seqcoding embedding, shape (hidden_dim,).
    Input must be a list containing a single (vh, vl) tuple.
    Returns result[0] to strip the batch dimension.
    """
    result = ablang_model([(vh, vl)], mode="seqcoding")  # numpy (1, d)
    return result[0]  # (d,)


# Smoke test
_emb = encode_antibody(
    "EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYWMSWVRQAPGK",
    "DIQMTQSPSSLSASVGDRVTITC",
)
EMBED_DIM = _emb.shape[0]
print(f"seqcoding output shape: {_emb.shape}  →  EMBED_DIM = {EMBED_DIM}")

Loading AbLang2 (downloads ~200 MB on first run)…
Mask character confirmed: '*'
AbLang2 parameters: 44.8M  |  all frozen ✓
seqcoding output shape: (480,)  →  EMBED_DIM = 480


## Section 4 — HCDR3 Identification

Key abnumber API facts (verified from v0.4.4 source):
- `Chain(seq, scheme='chothia')` — correct constructor
- `chain.cdr3_seq` — **direct property** returning the CDR3 string (no method call needed)
- There is **no** `cdr3_scheme_position()` or `is_cdr3()` method

In [5]:
def get_hcdr3(vh_seq: str, scheme: str = "chothia") -> Tuple[str, int, int]:
    """
    Returns (cdr3_seq, start_idx, end_idx) where end is exclusive.
    Uses chain.cdr3_seq property (the only correct abnumber API).
    Falls back to a length-based heuristic if numbering fails.
    """
    try:
        chain    = AbnumberChain(vh_seq, scheme=scheme)
        cdr3_seq = chain.cdr3_seq   # direct property, returns string

        if not cdr3_seq:
            raise ValueError("abnumber returned empty CDR3")

        start = vh_seq.find(cdr3_seq)
        if start == -1:
            raise ValueError(f"CDR3 '{cdr3_seq}' not found in VH")

        return cdr3_seq, start, start + len(cdr3_seq)

    except Exception as exc:
        warnings.warn(f"abnumber failed ({exc}); using positional heuristic.")
        # Heuristic: HCDR3 ≈ residues 95-105 in a ~120 aa VH
        s = int(0.79 * len(vh_seq))
        e = int(0.90 * len(vh_seq))
        return vh_seq[s:e], s, e


def mask_hcdr3(vh_seq: str) -> Tuple[str, int, int]:
    """Replace HCDR3 with AbLang2 mask character '*'."""
    cdr3, s, e = get_hcdr3(vh_seq)
    return vh_seq[:s] + ABLANG_MASK_CHAR * (e - s) + vh_seq[e:], s, e


# Test
_vh = (
    "EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYWMSWVRQAPGKGLEWVANIKQDGSEKYYVDSVKGRFTIS"
    "RDNAKNSLYLQMNSLRAEDTAVYYCARVGYYGSSYAMDYWGQGTLVTVSS"
)
_c, _s, _e = get_hcdr3(_vh)
print(f"VH length: {len(_vh)}")
print(f"HCDR3: '{_c}'  (pos {_s}–{_e})")
assert _c in _vh, "CDR3 not found in VH — check abnumber."

VH length: 121
HCDR3: 'CARVGYYGSSYAM'  (pos 95–108)


## Section 5 — Encode PROPHET-Ab

In [7]:
print(df_prophet)

    antibody_id antibody_name   Titer   Purity  SEC %Monomer    SMAC    HIC  \
0     GDPa1-001    abagovomab  140.25   98.530        97.010   2.730  2.590   
1     GDPa1-002    abituzumab  193.31   99.825        97.620   2.745  2.545   
2     GDPa1-003   abrezekimab  114.75   98.350        89.055   2.740  2.705   
3     GDPa1-004     abrilumab  327.32   98.575        98.605   2.715  2.565   
4     GDPa1-005    adalimumab  313.39   99.300        96.120   2.705  2.495   
..          ...           ...     ...      ...           ...     ...    ...   
241   GDPa1-242   visilizumab  303.51   94.700        98.110   2.745  2.495   
242   GDPa1-243    xentuzumab  281.21   98.970        99.875  10.000  4.500   
243   GDPa1-244   zalutumumab  266.55   89.430        98.065   2.705  2.590   
244   GDPa1-245   zanolimumab  301.96   99.115        97.780   2.725  2.660   
245   GDPa1-246  zolbetuximab  375.20  100.000        99.465   2.680  2.530   

       HAC    PR_CHO    PR_Ova  ...  \
0      NaN  

In [6]:
print("Encoding PROPHET-Ab with frozen AbLang2…")
emb_list, failed_ids = [], []

for _, row in tqdm(df_prophet.iterrows(), total=len(df_prophet)):
    try:
        emb_list.append(encode_antibody(row["vh_sequence"], row["vl_sequence"]))
    except Exception as exc:
        warnings.warn(f"{row['antibody_id']}: {exc}")
        failed_ids.append(row["antibody_id"])
        emb_list.append(np.zeros(EMBED_DIM))

X_all = np.vstack(emb_list)
fail_mask        = df_prophet["antibody_id"].isin(failed_ids).values
X_all            = X_all[~fail_mask]
df_prophet_clean = df_prophet[~fail_mask].reset_index(drop=True)

print(f"Embedding matrix: {X_all.shape}  |  failed: {len(failed_ids)}")

Encoding PROPHET-Ab with frozen AbLang2…


  0%|          | 0/246 [00:00<?, ?it/s]

Embedding matrix: (0, 480)  |  failed: 246


## Section 6 — Ridge Regression Developability Predictors

In [ ]:
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)

def fit_developability_predictor(
    X: np.ndarray, y: np.ndarray, name: str, n_splits: int = 5,
) -> Tuple[Pipeline, np.ndarray, np.ndarray]:
    valid = ~np.isnan(y)
    Xv, yv = X[valid], y[valid]
    cv   = KFold(n_splits=min(n_splits, len(yv)), shuffle=True, random_state=SEED)
    pipe = Pipeline([("sc", StandardScaler()), ("ridge", RidgeCV(alphas=ALPHAS, cv=cv))])
    y_cv = cross_val_predict(pipe, Xv, yv, cv=cv)
    r2   = r2_score(yv, y_cv)
    rmse = np.sqrt(mean_squared_error(yv, y_cv))
    pipe.fit(Xv, yv)
    print(f"{name:22s}  R²(CV)={r2:+.3f}  RMSE(CV)={rmse:.3f}  n={valid.sum()}")
    return pipe, yv, y_cv

print("Ridge Regression Developability Predictors (AbLang2 features)")
print("=" * 58)
ridge_hic,    y_hic_v,    y_hic_cv    = fit_developability_predictor(X_all, df_prophet_clean["hic_rt"].values.astype(float),    "HIC Retention Time")
ridge_acsins, y_acsins_v, y_acsins_cv = fit_developability_predictor(X_all, df_prophet_clean["ac_sins"].values.astype(float),   "AC-SINS")

_hic_mu,    _hic_sig    = y_hic_v.mean(),    y_hic_v.std()    + 1e-8
_acsins_mu, _acsins_sig = y_acsins_v.mean(), y_acsins_v.std() + 1e-8

def score_developability(X_emb: np.ndarray) -> np.ndarray:
    """Higher = more developable (lower predicted HIC and AC-SINS)."""
    return -((ridge_hic.predict(X_emb) - _hic_mu) / _hic_sig +
             (ridge_acsins.predict(X_emb) - _acsins_mu) / _acsins_sig)

# CV scatter plots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, yt, yp, name in zip(axes,
    [y_hic_v, y_acsins_v], [y_hic_cv, y_acsins_cv],
    ["HIC Retention Time", "AC-SINS"]):
    ax.scatter(yt, yp, alpha=0.6, s=20)
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set(xlabel=f"True {name}", ylabel="Predicted (CV)",
           title=f"{name}\nR²={r2_score(yt,yp):.3f}")
plt.tight_layout(); plt.savefig("ridge_cv.png", dpi=150); plt.show()

## Section 7 — OAS Data Loading

Download paired bulk CSVs from https://opig.stats.ox.ac.uk/webapps/oas/oas_paired/  
(Filter: Species=Human, Isotype=IgG) and place them in `oas_cache/`.  
For production use `OAS_MAX_SEQS = 500_000` and `N_EPOCHS_FLOW = 100`.

In [ ]:
import gzip, csv, urllib.request

OAS_SAMPLE_URLS: List[str] = [
    # Add real OAS bulk CSV URLs here, e.g.:
    # "https://opig.stats.ox.ac.uk/.../SRR7959893_paired.csv.gz",
]
OAS_MAX_SEQS = 5_000  # use 500_000 for real training


def load_oas_paired(urls: List[str], max_seqs: int = OAS_MAX_SEQS) -> pd.DataFrame:
    os.makedirs("oas_cache", exist_ok=True)
    rows: List[Dict] = []
    for url in urls:
        fname = os.path.join("oas_cache", os.path.basename(url))
        if not os.path.exists(fname):
            print(f"Downloading {url}…")
            try: urllib.request.urlretrieve(url, fname)
            except Exception as e: print(f"  ✗ {e}"); continue
        try:
            with gzip.open(fname, "rt") as fh:
                next(fh)  # skip OAS metadata header line
                for r in csv.DictReader(fh):
                    vh = r.get("sequence_alignment_aa_heavy", "").replace("-", "")
                    vl = r.get("sequence_alignment_aa_light", "").replace("-", "")
                    if len(vh) >= 50 and len(vl) >= 50:
                        rows.append({"vh_sequence": vh, "vl_sequence": vl})
                    if len(rows) >= max_seqs: break
        except Exception as e: print(f"  ✗ Parse error: {e}")
        if len(rows) >= max_seqs: break
    return pd.DataFrame(rows)


def make_synthetic_oas(n: int = 5_000) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    def rseq(lo, hi): return "".join(rng.choice(AA_VOCAB, rng.integers(lo, hi)))
    return pd.DataFrame([{"vh_sequence": rseq(115, 130), "vl_sequence": rseq(105, 115)}
                          for _ in range(n)])

df_oas = load_oas_paired(OAS_SAMPLE_URLS)
if len(df_oas) < 100:
    print("[DEMO] OAS unavailable — using synthetic sequences.")
    df_oas = make_synthetic_oas()
print(f"OAS sequences: {len(df_oas):,}")

## Section 8 — Flow Matching Network (OT-CFM)

In [ ]:
def seq_to_onehot(seq: str, length: int = MAX_CDR3_LEN) -> torch.Tensor:
    oh = torch.zeros(length, N_AA)
    for i, aa in enumerate(seq[:length]):
        if aa in AA_TO_IDX:
            oh[i, AA_TO_IDX[aa]] = 1.0
    return oh


def onehot_to_seq(oh: torch.Tensor, max_len: Optional[int] = None) -> str:
    """Greedy decode (L, N_AA) logits → string; stops at all-zero padding rows."""
    out, L = [], oh.shape[0] if max_len is None else min(max_len, oh.shape[0])
    for i in range(L):
        if oh[i].sum().item() < 1e-6: break
        out.append(IDX_TO_AA.get(int(oh[i].argmax()), ""))
    return "".join(out)


class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__(); self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:  # t: (B,) → (B, dim)
        half  = self.dim // 2
        freqs = torch.exp(-math.log(10_000) * torch.arange(half, device=t.device).float() / (half - 1))
        args  = t[:, None] * freqs[None, :]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class FlowMatchingNet(nn.Module):
    """
    Velocity-field network v_θ(x_t, t, context).
    Three tokens fed to a TransformerEncoder: [x_token, ctx_token, t_token].
    Output velocity is read from x_token.
    """
    def __init__(self, context_dim: int, d_model: int = 256,
                 n_heads: int = 4, n_layers: int = 4, dropout: float = 0.1):
        super().__init__()
        flat = MAX_CDR3_LEN * N_AA
        self.x_proj   = nn.Sequential(nn.Linear(flat, d_model), nn.SiLU())
        self.ctx_proj = nn.Linear(context_dim, d_model)
        self.time_emb = SinusoidalTimeEmb(d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.out = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, flat))

    def forward(self, x_t: torch.Tensor, t: torch.Tensor,
                context: torch.Tensor) -> torch.Tensor:
        # x_t: (B, L, 20)  t: (B,)  context: (B, d)
        B = x_t.shape[0]
        tokens = torch.stack([
            self.x_proj(x_t.reshape(B, -1)),
            self.ctx_proj(context),
            self.time_emb(t),
        ], dim=1)                                        # (B, 3, d_model)
        tokens = self.transformer(tokens)
        return self.out(tokens[:, 0]).reshape(B, MAX_CDR3_LEN, N_AA)


# Sanity check
_m = FlowMatchingNet(EMBED_DIM).to(DEVICE)
_o = _m(torch.randn(4, MAX_CDR3_LEN, N_AA, device=DEVICE),
         torch.rand(4, device=DEVICE),
         torch.randn(4, EMBED_DIM, device=DEVICE))
print(f"FlowMatchingNet output: {tuple(_o.shape)} ✓  "
      f"({sum(p.numel() for p in _m.parameters())/1e6:.2f}M params)")
del _m, _o

## Section 9 — OAS Dataset & Training

In [ ]:
class OASFlowDataset(Dataset):
    """Pre-computes (HCDR3 one-hot, masked-antibody AbLang2 embedding) pairs."""

    def __init__(self, df: pd.DataFrame, max_samples: Optional[int] = None):
        self.items: List[Dict] = []
        df_sub = df.iloc[:max_samples] if max_samples else df
        print(f"Pre-computing embeddings for {len(df_sub)} OAS sequences…")
        skipped = 0
        for _, row in tqdm(df_sub.iterrows(), total=len(df_sub)):
            try:
                vh, vl      = row["vh_sequence"], row["vl_sequence"]
                cdr3, s, e  = get_hcdr3(vh)
                if not cdr3: skipped += 1; continue
                vh_masked   = vh[:s] + ABLANG_MASK_CHAR * (e - s) + vh[e:]
                ctx         = encode_antibody(vh_masked, vl)
                self.items.append({
                    "x1":      seq_to_onehot(cdr3, MAX_CDR3_LEN),
                    "context": torch.from_numpy(ctx).float(),
                })
            except Exception: skipped += 1
        print(f"  kept {len(self.items):,}  |  skipped {skipped}")

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


OAS_TRAIN_SIZE = 2_000   # set to 500_000 for real training
oas_ds     = OASFlowDataset(df_oas, max_samples=OAS_TRAIN_SIZE)
train_dl   = DataLoader(oas_ds, batch_size=32, shuffle=True, num_workers=0,
                        pin_memory=(DEVICE.type == "cuda"))
print(f"DataLoader: {len(oas_ds):,} samples, {len(train_dl)} batches/epoch")

In [ ]:
def train_flow_model(
    model: FlowMatchingNet, loader: DataLoader,
    n_epochs: int = 30, lr: float = 3e-4, sigma_min: float = 0.01,
) -> List[float]:
    """
    OT-CFM training loop:
      x_t = (1−t)·x₀ + t·x₁  ,  u_t = x₁ − x₀
      L   = ‖v_θ(x_t, t, c) − u_t‖²
    """
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs*len(loader))
    model.train()
    losses: List[float] = []

    for ep in range(n_epochs):
        ep_loss = 0.0
        for batch in loader:
            x1      = batch["x1"].to(DEVICE)      # (B, L, 20)
            context = batch["context"].to(DEVICE)  # (B, d)
            B       = x1.shape[0]
            x0      = torch.randn_like(x1) * sigma_min
            t       = torch.rand(B, device=DEVICE)
            x_t     = (1 - t[:, None, None]) * x0 + t[:, None, None] * x1
            loss    = F.mse_loss(model(x_t, t, context), x1 - x0)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            ep_loss += loss.item()
        avg = ep_loss / len(loader); losses.append(avg)
        if (ep + 1) % 5 == 0:
            print(f"Epoch {ep+1:3d}/{n_epochs}  loss={avg:.4f}  lr={sched.get_last_lr()[0]:.2e}")
    return losses


N_EPOCHS_FLOW = 30  # set to 100+ for real training
flow_model    = FlowMatchingNet(EMBED_DIM).to(DEVICE)
print(f"Training OT-CFM for {N_EPOCHS_FLOW} epochs…")
loss_hist = train_flow_model(flow_model, train_dl, n_epochs=N_EPOCHS_FLOW)
torch.save({"state": flow_model.state_dict(), "embed_dim": EMBED_DIM,
            "losses": loss_hist}, "flow_model.pt")

plt.figure(figsize=(7,3)); plt.plot(loss_hist)
plt.xlabel("Epoch"); plt.ylabel("CFM Loss"); plt.title("Flow Matching Training")
plt.tight_layout(); plt.savefig("flow_loss.png", dpi=150); plt.show()

## Section 10 — Inference: Beam-Guided HCDR3 Infilling

In [ ]:
@torch.no_grad()
def score_candidate(vh_gen: str, vl: str) -> float:
    try:
        return float(score_developability(encode_antibody(vh_gen, vl).reshape(1, -1))[0])
    except Exception:
        return float("-inf")


@torch.no_grad()
def beam_guided_infill(
    vh_parent: str, vl_parent: str, model: FlowMatchingNet,
    k_beams: int = 5, n_steps: int = 20, score_every: int = 5,
    temperature: float = 1.0,
) -> List[Dict]:
    """
    Beam-guided Euler integration of the flow ODE for HCDR3 infilling.

    1. Encode masked VH → context c
    2. Sample k noise vectors x₀
    3. Euler step: x ← x + (1/n_steps) · v_θ(x, t, c)
    4. Every score_every steps: keep best beam, re-fan with small noise
    5. Decode & splice CDR3 back into framework
    """
    model.eval()
    cdr3_orig, s, e = get_hcdr3(vh_parent)
    cdr3_len        = min(len(cdr3_orig), MAX_CDR3_LEN)

    vh_masked = vh_parent[:s] + ABLANG_MASK_CHAR * (e - s) + vh_parent[e:]
    ctx = torch.from_numpy(encode_antibody(vh_masked, vl_parent)).float().to(DEVICE)
    ctx = ctx.unsqueeze(0).expand(k_beams, -1)  # (k, EMBED_DIM)

    x  = torch.randn(k_beams, MAX_CDR3_LEN, N_AA, device=DEVICE)
    dt = 1.0 / n_steps

    for step in range(n_steps):
        t_val = torch.full((k_beams,), step * dt, device=DEVICE)
        x     = x + dt * model(x, t_val, ctx)

        if (step + 1) % score_every == 0:
            scores = []
            for bi in range(k_beams):
                cdr3_c = onehot_to_seq(F.softmax(x[bi]/temperature, dim=-1), cdr3_len)
                if not cdr3_c: scores.append(float("-inf")); continue
                scores.append(score_candidate(
                    vh_parent[:s] + cdr3_c + vh_parent[e:], vl_parent))
            best = int(np.argmax(scores))
            x    = x[best].unsqueeze(0).expand(k_beams, -1, -1).clone()
            x   += torch.randn_like(x) * 0.05

    results = []
    for bi in range(k_beams):
        cdr3_g = onehot_to_seq(F.softmax(x[bi]/temperature, dim=-1), cdr3_len)
        if not cdr3_g: continue
        vh_g = vh_parent[:s] + cdr3_g + vh_parent[e:]
        results.append({"vh_generated": vh_g, "cdr3_generated": cdr3_g,
                        "cdr3_original": cdr3_orig[:cdr3_len],
                        "dev_score": score_candidate(vh_g, vl_parent)})
    results.sort(key=lambda d: d["dev_score"], reverse=True)
    return results


print("Beam-guided inference defined ✓")

In [ ]:
N_INFERENCE = 10; K_BEAMS = 5; N_RUNS = 3

print(f"Infilling on {N_INFERENCE} antibodies ({N_RUNS} runs × {K_BEAMS} beams)…")
all_results: List[Dict] = []

for _, row in tqdm(df_prophet_clean.iloc[:N_INFERENCE].iterrows(), total=N_INFERENCE):
    for run_id in range(N_RUNS):
        try:
            cands = beam_guided_infill(
                row["vh_sequence"], row["vl_sequence"], flow_model,
                k_beams=K_BEAMS, n_steps=20, score_every=5,
            )
            if cands:
                b = cands[0]
                all_results.append({
                    "antibody_id": row["antibody_id"], "run_id": run_id,
                    "vh_parent":      row["vh_sequence"],
                    "vh_generated":   b["vh_generated"],
                    "cdr3_original":  b["cdr3_original"],
                    "cdr3_generated": b["cdr3_generated"],
                    "dev_score":      b["dev_score"],
                })
        except Exception as exc:
            warnings.warn(f"{row['antibody_id']} run {run_id}: {exc}")

df_gen = pd.DataFrame(all_results)
print(f"Generated {len(df_gen)} sequences.")
print(df_gen[["antibody_id","cdr3_original","cdr3_generated","dev_score"]].head())

## Section 11 — Evaluation Metrics

### 11.1 Sequence Metrics

In [ ]:
from difflib import SequenceMatcher

def seq_id(a, b):  return SequenceMatcher(None, a, b).ratio() if a and b else 0.0
def hamming(a, b): return sum(x!=y for x,y in zip(a[:min(len(a),len(b))], b[:min(len(a),len(b))]))

def camsol_approx(seq: str) -> float:
    """Simplified CamSol score (Sormanni et al. JMB 2015). + = more soluble."""
    W = {"A": 0.06,"R": 0.89,"N": 0.20,"D": 0.39,"C":-0.76,"E": 0.26,"Q": 0.21,
         "G":-0.26,"H": 0.49,"I":-0.83,"L":-0.79,"K": 0.77,"M":-0.69,"F":-1.06,
         "P":-0.36,"S": 0.17,"T": 0.11,"W":-1.37,"Y":-0.42,"V":-0.65}
    if not seq: return 0.0
    v = [W.get(a.upper(), 0.0) for a in seq]; w=9
    return float(np.mean([np.mean(v[max(0,i-w//2):i+w//2+1]) for i in range(len(v))]))

def sap_approx(seq: str) -> float:
    """Sequence SAP proxy (Chennamsetty et al. PNAS 2009). Higher = more aggregation risk."""
    W = {"A": 0.0,"R":-6.0,"N":-3.5,"D":-6.0,"C": 4.5,"E":-6.0,"Q":-3.5,
         "G": 0.0,"H":-3.2,"I": 4.5,"L": 3.8,"K":-6.0,"M": 1.9,"F": 2.8,
         "P": 0.0,"S":-0.8,"T":-0.7,"W": 1.9,"Y": 1.3,"V": 4.2}
    if not seq: return 0.0
    return float(np.mean([W.get(a.upper(),0.0) for a in seq]))


if len(df_gen) > 0:
    df_gen["seq_id_cdr3"]     = df_gen.apply(lambda r: seq_id(r.cdr3_original, r.cdr3_generated), axis=1)
    df_gen["hamming_cdr3"]    = df_gen.apply(lambda r: hamming(r.cdr3_original, r.cdr3_generated), axis=1)
    df_gen["camsol_parent"]   = df_gen["vh_parent"].apply(camsol_approx)
    df_gen["camsol_gen"]      = df_gen["vh_generated"].apply(camsol_approx)
    df_gen["sap_parent"]      = df_gen["cdr3_original"].apply(sap_approx)
    df_gen["sap_gen"]         = df_gen["cdr3_generated"].apply(sap_approx)
    df_gen["delta_camsol"]    = df_gen["camsol_gen"]  - df_gen["camsol_parent"]
    df_gen["delta_sap"]       = df_gen["sap_gen"]     - df_gen["sap_parent"]

    print(df_gen[["dev_score","seq_id_cdr3","delta_camsol","delta_sap"]].describe().round(3))

### 11.2 IgFold + FoldX ΔΔG (Optional)

Download FoldX (free academic license) from https://foldxsuite.crg.eu  
and place binary at `/content/foldx/foldx`.

In [ ]:
import subprocess, tempfile, shutil
FOLDX_BIN = "/content/foldx/foldx"


def predict_igfold(vh: str, vl: str, out_pdb: str) -> Optional[str]:
    if not IGFOLD_AVAILABLE: return None
    try:
        IgFoldRunner().fold(out_pdb, sequences={"H": vh, "L": vl})
        return out_pdb if os.path.exists(out_pdb) else None
    except Exception as e: warnings.warn(f"IgFold: {e}"); return None


def foldx_stability(pdb: str) -> Optional[float]:
    if not os.path.exists(FOLDX_BIN): return None
    td = tempfile.mkdtemp()
    try:
        name = os.path.basename(pdb); shutil.copy(pdb, os.path.join(td, name))
        subprocess.run([FOLDX_BIN, "--command=Stability", f"--pdb={name}"],
                       cwd=td, capture_output=True, timeout=180, check=False)
        st = os.path.join(td, name.replace(".pdb", "_ST.fxout"))
        if not os.path.exists(st): return None
        with open(st) as fh:
            for line in fh:
                parts = line.strip().split()
                if parts and not parts[0].startswith("#") and len(parts) >= 2:
                    return float(parts[1])
    except Exception as e: warnings.warn(f"FoldX: {e}")
    finally: shutil.rmtree(td, ignore_errors=True)
    return None


RUN_STRUCT = IGFOLD_AVAILABLE and os.path.exists(FOLDX_BIN)
df_gen["ddG"] = np.nan

if RUN_STRUCT and len(df_gen) > 0:
    os.makedirs("structures", exist_ok=True)
    for _, row in tqdm(df_gen.drop_duplicates("antibody_id").head(5).iterrows()):
        vl = df_prophet_clean.loc[
            df_prophet_clean.antibody_id == row.antibody_id, "vl_sequence"
        ].values[0]
        p_pdb = f"structures/{row.antibody_id}_parent.pdb"
        g_pdb = f"structures/{row.antibody_id}_gen.pdb"
        if predict_igfold(row.vh_parent, vl, p_pdb) and predict_igfold(row.vh_generated, vl, g_pdb):
            dg_p, dg_g = foldx_stability(p_pdb), foldx_stability(g_pdb)
            if dg_p and dg_g:
                df_gen.loc[df_gen.antibody_id == row.antibody_id, "ddG"] = dg_g - dg_p
    print(df_gen[["antibody_id","ddG"]].dropna())
else:
    print("Structure ΔΔG skipped. Enable by installing igfold + downloading FoldX.")

### 11.3 Summary Plots

In [ ]:
if len(df_gen) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    specs = [("dev_score","steelblue","Developability Score"),
             ("seq_id_cdr3","coral","HCDR3 Sequence Identity"),
             ("delta_camsol","mediumseagreen","ΔCamSol (gen−parent)"),
             ("delta_sap","mediumpurple","ΔSAP (gen−parent)")]
    for ax, (col, col_, label) in zip(axes, specs):
        v = df_gen[col].dropna()
        ax.hist(v, bins=15, color=col_, edgecolor="white")
        ax.axvline(v.mean(), c="black", ls="--", lw=1.2, label=f"μ={v.mean():.2f}")
        if "delta" in col: ax.axvline(0, c="red", ls=":", lw=1)
        ax.set_xlabel(label); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig("generation_metrics.png", dpi=150); plt.show()
    print(df_gen[["dev_score","seq_id_cdr3","delta_camsol","delta_sap"]].describe().round(3))

## Section 12 — Save Outputs

In [ ]:
df_gen.to_csv("generated_antibodies.csv", index=False)
with open("developability_predictors.pkl", "wb") as fh:
    pickle.dump({"hic": ridge_hic, "ac_sins": ridge_acsins}, fh)
print("Saved: generated_antibodies.csv  |  flow_model.pt  |  developability_predictors.pkl")

---
## Section 13 — Diffusion Baseline: EvoDiff OA-DM-38M

**Why EvoDiff instead of ABDiffusion:**  
ABDiffusion's API is currently unavailable. EvoDiff (Alamdari et al. *Nat Comm* 2024) uses the same category of model — **Order-Agnostic Masked Discrete Language Model (MDLM)** — operating on protein token sequences directly. It is a valid like-for-like comparison (discrete diffusion) against our continuous flow matching approach.

**Verified API facts from evodiff v1.1.2 source:**
- `OA_DM_38M()` returns **4 values**: `(model, collater, tokenizer, scheme)` — unpack all 4
- `inpaint_simple(model, sequence, start_idx, end_idx, tokenizer, device)` — first arg is **model**, not the full tuple
- Returns `(token_tensor, full_vh_str, cdr3_str)` — the full generated sequence and CDR3 region as strings
- Import path: `from evodiff.conditional_generation import inpaint_simple`

In [ ]:
def evodiff_infill_hcdr3(vh: str, vl: str, n_samples: int = 5) -> List[Dict]:
    """
    EvoDiff HCDR3 infilling baseline.

    IMPORTANT:
    - OA_DM_38M() → (model, collater, tokenizer, scheme)   [4 values]
    - inpaint_simple(model, seq, start, end, tokenizer, device) → (tokens, full_seq, cdr3_str)
    - EvoDiff operates on VH only (single-chain model)
    """
    if not EVODIFF_AVAILABLE:
        return []

    # Unpack all 4 return values
    model_evo, _collater, tokenizer_evo, _scheme = OA_DM_38M()
    model_evo = model_evo.to(DEVICE).eval()

    cdr3_orig, start, end = get_hcdr3(vh)
    results: List[Dict] = []

    for _ in range(n_samples):
        try:
            # inpaint_simple: first arg is model (not tuple), returns 3 values
            _tokens, full_vh_gen, cdr3_gen = inpaint_simple(
                model_evo, vh,
                start_idx=start, end_idx=end,
                tokenizer=tokenizer_evo,
                device=str(DEVICE),
            )
            ds = score_candidate(full_vh_gen, vl)
            results.append({
                "vh_generated":   full_vh_gen,
                "cdr3_generated": cdr3_gen,
                "cdr3_original":  cdr3_orig,
                "dev_score":      ds,
                "delta_camsol":   camsol_approx(full_vh_gen) - camsol_approx(vh),
                "delta_sap":      sap_approx(cdr3_gen)       - sap_approx(cdr3_orig),
            })
        except Exception as exc:
            warnings.warn(f"EvoDiff inpaint: {exc}")

    results.sort(key=lambda d: d["dev_score"], reverse=True)
    return results


if EVODIFF_AVAILABLE:
    print(f"Running EvoDiff OA-DM-38M on {N_INFERENCE} antibodies…")
    evo_rows: List[Dict] = []
    for _, row in tqdm(df_prophet_clean.iloc[:N_INFERENCE].iterrows(), total=N_INFERENCE):
        cands = evodiff_infill_hcdr3(row["vh_sequence"], row["vl_sequence"], n_samples=K_BEAMS)
        if cands:
            b = cands[0]
            evo_rows.append({"antibody_id": row["antibody_id"],
                             "model": "EvoDiff-OA-DM-38M",
                             "dev_score": b["dev_score"],
                             "delta_camsol": b["delta_camsol"],
                             "delta_sap": b["delta_sap"]})

    df_evo = pd.DataFrame(evo_rows)

    # Compare
    df_flow_top = (df_gen.groupby("antibody_id")["dev_score"]
                   .max().reset_index().assign(model="FlowMatch-OT-CFM"))
    df_cmp = pd.concat([df_flow_top[["model","dev_score"]],
                        df_evo[["model","dev_score"]]], ignore_index=True)

    plt.figure(figsize=(7,4))
    for m, g in df_cmp.groupby("model"):
        plt.hist(g["dev_score"], bins=12, alpha=0.65, label=m)
    plt.xlabel("Developability Score")
    plt.title("Flow Matching vs EvoDiff: Developability Scores")
    plt.legend(); plt.tight_layout()
    plt.savefig("model_comparison.png", dpi=150); plt.show()
    print(df_cmp.groupby("model")["dev_score"].agg(["mean","std","count"]).round(3))
else:
    print("EvoDiff baseline skipped. Install: pip install evodiff")

---
## Quick Reference: Real Data & Scaling

| What | How |
|------|-----|
| **PROPHET-Ab data** | Download supp. tables from https://doi.org/10.1080/19420862.2025.2593055 → `prophet_ab.csv` |
| **OAS sequences** | Bulk paired CSVs at https://opig.stats.ox.ac.uk/webapps/oas/oas_paired/ → `oas_cache/` |
| **Training scale** | `OAS_TRAIN_SIZE = 500_000`, `N_EPOCHS_FLOW = 100` |
| **Inference scale** | `N_INFERENCE = len(df_prophet_clean)`, `N_RUNS = 10` |
| **FoldX** | Academic license at https://foldxsuite.crg.eu → `/content/foldx/foldx` |
| **DeepSP (3D SAP)** | `git clone https://github.com/PKLPKL/DeepSP` — requires IgFold structure first |
| **Speed** | `torch.compile(flow_model)` gives ~2× speedup on PyTorch ≥ 2.0 |